In [1]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Database connection configuration to SAKIP
DB_HOST = os.getenv('DB_HOST_SIMPEG', 'localhost')
DB_PORT = os.getenv('DB_PORT_SIMPEG', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_SIMPEG', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_SIMPEG', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_SIMPEG', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_simpeg = create_engine(connection_string, echo=False)
    with engine_simpeg.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: simpeg_jabar on 10.110.32.121:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [3]:
def load_data_from_sql(query, engine):
    """
    Load data from PostgreSQL database into a pandas DataFrame.
    
    Parameters:
    query (str): SQL query to execute
    engine: SQLAlchemy engine object
    
    Returns:
    pandas.DataFrame: Data from the query
    """
    connection = None
    try:
        # Create a new connection and rollback any pending transaction
        connection = engine.connect()
        
        # Rollback any pending transaction to ensure clean state
        try:
            connection.rollback()
        except:
            pass  # If no transaction to rollback, ignore
        
        # Execute the query
        df = pd.read_sql(query, connection)
        print(f"✅ Data loaded successfully! Shape: {df.shape}")
        
        connection.close()
        return df
    except Exception as e:
        # Ensure connection is closed on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        print(f"❌ Error loading data: {e}")
        return None

def get_table_info(table_name, engine):
    """
    Get basic information about a table.
    
    Parameters:
    table_name (str): Name of the table
    engine: SQLAlchemy engine object
    """
    try:
        # Get table structure using PostgreSQL information_schema
        structure_query = f"""
            SELECT 
                column_name,
                data_type,
                character_maximum_length,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_name = '{table_name}'
            ORDER BY ordinal_position
        """
        structure = pd.read_sql(structure_query, engine)
        
        # Get row count
        count_query = f"SELECT COUNT(*) as row_count FROM {table_name}"
        count_result = pd.read_sql(count_query, engine)
        
        print(f"📊 Table: {table_name}")
        print(f"Rows: {count_result['row_count'].iloc[0]}")
        print(f"Columns: {len(structure)}")
        print("\nColumn Information:")
        print(structure)
        
        return structure
    except SQLAlchemyError as e:
        print(f"❌ Error getting table info: {e}")
        return None

def list_tables(engine):
    """
    List all tables in the database.
    
    Parameters:
    engine: SQLAlchemy engine object
    """
    try:
        # Use PostgreSQL information_schema to list tables
        query = """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            ORDER BY table_name
        """
        tables = pd.read_sql(query, engine)
        print("📋 Available tables:")
        for table in tables['table_name']:
            print(f"  - {table}")
        return tables
    except SQLAlchemyError as e:
        print(f"❌ Error listing tables: {e}")
        return None

print("Data loading functions defined successfully!")



Data loading functions defined successfully!


In [4]:
# Load data from pkl

df_vpd_akhir_tahun = pd.read_pickle('df_vpd_akhir_tahun.pkl')
df_vpd_tw4 = pd.read_pickle('df_vpd_tw4.pkl')

# Remove rows where peg_status = false
df_vpd_akhir_tahun = df_vpd_akhir_tahun[df_vpd_akhir_tahun['peg_status'] == True]
df_vpd_tw4 = df_vpd_tw4[df_vpd_tw4['peg_status'] == True]

# df_vpd_akhir_tahun.head()

In [5]:
# Database connection configuration to SAKIP
DB_HOST = os.getenv('DB_HOST_2025', 'localhost')
DB_PORT = os.getenv('DB_PORT_2025', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_2025', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_2025', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_2025', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_2025 = create_engine(connection_string, echo=False)
    with engine_2025.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")

Connecting to database: erk_ekinerja_2025 on 10.110.32.114:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [6]:
def build_monitoring_table(df_pkl, engine, alasan='tahunan'):

    # Build Monitoring Table
    df_monitoring = pd.DataFrame()
    df_monitoring = df_pkl[['satuan_kerja_id', 'satuan_kerja_nama', 
    'unit_kerja_id', 'unit_kerja_nama', 'peg_status_kepegawaian',
    'peg_nip', 'peg_nama', 'jabatan_jenis', 'eselon_nm',
    'jabatan_nama', 'tugas_tambahan_jabatan_nama',
    'is_gtk', 'unit_kerja_parent_nama',
    'unit_kerja_nama_full', 'peg_lahir_tanggal', 'peg_umur_pensiun',
    'nip_atasan', 'nama_atasan']]

    # Order by satuan_kerja_id then unit_kerja_id
    df_monitoring = df_monitoring.sort_values(by=['satuan_kerja_id', 'unit_kerja_id', 'jabatan_jenis'])


    # Get atasan jabatan_nama
    df_monitoring['jabatan_jenis'] = df_monitoring['jabatan_jenis'].astype(int).astype(str)
    # Get atasan name: if jabatan_jenis == '2' (i.e., eselon), get jabatan_nama; else, tugas_tambahan_jabatan_nama
    # Create lookup dictionaries for better performance
    nip_to_jabatan_nama = df_monitoring.set_index('peg_nip')['jabatan_nama'].to_dict()
    nip_to_tugas_tambahan = df_monitoring.set_index('peg_nip')['tugas_tambahan_jabatan_nama'].to_dict()
    nip_to_jabatan_jenis = df_monitoring.set_index('peg_nip')['jabatan_jenis'].to_dict()

    # Get atasan jabatan using vectorized operations where possible
    df_monitoring['jabatan_atasan'] = df_monitoring['nip_atasan'].apply(
        lambda nip_atasan: (
            nip_to_jabatan_nama.get(nip_atasan) 
            if str(nip_to_jabatan_jenis.get(nip_atasan, '')) == '2' 
            else nip_to_tugas_tambahan.get(nip_atasan)
        ) if nip_atasan in nip_to_jabatan_jenis else None
    )

    # Map jabatan_jenis
    df_monitoring['jabatan_jenis'] = df_monitoring.apply(
        lambda row: row['eselon_nm'] if row['jabatan_jenis'] == '2' else (
            'Fungsional Tertentu' if row['jabatan_jenis'] == '3' else (
                'Fungsional Umum' if row['jabatan_jenis'] == '4' else row['jabatan_jenis']
            )
        ),
        axis=1
    )

    # Exclude gubernur and wakil
    df_monitoring = df_monitoring[~df_monitoring['peg_nip'].isin(['3000000002', '3000000001'])]

    # Get tmt_pensiun peg_lahir_tanggal + peg_umur_pensiun, first day of next month
    df_monitoring['peg_lahir_tanggal'] = pd.to_datetime(df_monitoring['peg_lahir_tanggal'], errors='coerce')
    df_monitoring['peg_umur_pensiun'] = pd.to_numeric(df_monitoring['peg_umur_pensiun'], errors='coerce')
    # Calculate tmt_pensiun using vectorized operations
    pensiun_dates = df_monitoring['peg_lahir_tanggal'] + pd.to_timedelta(df_monitoring['peg_umur_pensiun'] * 365.25, unit='D')
    next_month_dates = (pensiun_dates + pd.DateOffset(months=1)).apply(lambda x: x.replace(day=1) if pd.notnull(x) else pd.NaT)
    df_monitoring['tmt_pensiun'] = next_month_dates.dt.strftime('%d-%m-%Y')

    # Get KCD
    df_monitoring_kcd = df_monitoring.copy()
    df_monitoring_kcd['unit_kerja_nama_full_split'] = df_monitoring_kcd['unit_kerja_nama_full'].str.split('|').apply(
        lambda parts: next((part for part in parts if 'CABANG PENDIDIKAN WILAYAH' in part), None) if isinstance(parts, list) else df_monitoring_kcd['satuan_kerja_nama']
    )
    df_monitoring_kcd['kcd'] = df_monitoring_kcd['unit_kerja_nama_full_split']

    # Drop columns
    df_monitoring_kcd.drop(columns=['eselon_nm','unit_kerja_nama_full_split','unit_kerja_parent_nama','unit_kerja_nama_full', 'peg_lahir_tanggal', 'peg_umur_pensiun'], inplace=True)

    # Move kcd to after is_gtk
    df_monitoring_kcd.insert(df_monitoring_kcd.columns.get_loc('is_gtk') + 1, 'kcd', df_monitoring_kcd.pop('kcd'))

    # Build Predikat columns
    df_final_predikat = get_predikat(df_monitoring_kcd, engine_2025, alasan)

    # Build SKP columns
    df_final = get_skp_data(df_final_predikat, engine_2025, alasan)

    # Compare nip_atasan with nip_atasan_skp
    df_final['same_atasan'] = df_final['nip_atasan'] == df_final['nip_atasan_skp']

    return df_final

def get_predikat(df_monitoring, engine, alasan='tahunan'):
    query = f"""
        SELECT *
        FROM nilai_skp_pegawai
        WHERE periode = '{alasan}'
    """
    df_predikat = pd.read_sql(query, engine)

    df_predikat = df_predikat.rename(columns={'submit': 'predikat_submit', 'peg_nip': 'nip'})

    df_predikat_slim = df_predikat[['nip', 'predikat_submit']].drop_duplicates()

    df_final = df_monitoring.merge(df_predikat_slim, left_on='peg_nip', right_on='nip', how='left')

    df_final = df_final.drop(columns=['nip'], errors='ignore')

    return df_final

def get_skp_data(df_monitoring, engine, alasan='tahunan'):
    query = f"""
        SELECT nip, id_sasaran, tanggal_tutup, nip_atasan, tanggal_export,
               ekspektasi_id, lampiran_id, generated_url
        FROM sasaran_tutup
        WHERE alasan = '{alasan}'
    """
    df_skp = pd.read_sql(query, engine)

    query = """
        SELECT id_referensi, status_realisasi, umpan_balik
        FROM sasaran_kinerja_realisasi
        WHERE deleted_at IS NULL
    """
    df_realisasi = pd.read_sql(query, engine)

    query = "SELECT * FROM lampiran_skp"
    df_lampiran = pd.read_sql(query, engine)

    query = "SELECT * FROM ekspektasi_perilaku"
    df_ekspektasi = pd.read_sql(query, engine)

    # Rename nip_atasan to nip_atasan_skp
    df_skp.rename(columns={'nip_atasan': 'nip_atasan_skp'}, inplace=True)

    # --- UB Kinerja ---
    df_skp_combine = df_skp[['nip', 'id_sasaran']].copy()
    df_skp_combine = df_skp_combine.explode('id_sasaran')
    df_skp_combine = df_skp_combine.merge(df_realisasi, left_on='id_sasaran', right_on='id_referensi', how='left')

    df_skp_count = df_skp_combine.groupby('nip').agg(
        count_sasaran=('id_sasaran', 'size'),
        count_umpan_balik=('umpan_balik', 'count'),
        count_status_realisasi=('status_realisasi', lambda x: (x == 1).sum())
    ).reset_index()
    df_skp_count['all_realisasi_valid'] = df_skp_count['count_status_realisasi'] == df_skp_count['count_sasaran']
    df_skp_count['has_ub_kinerja'] = df_skp_count['count_umpan_balik'] == df_skp_count['count_sasaran']

    # --- Lampiran ---
    lampiran_cols = ['dukungan_sumber_daya', 'skema_pertanggungjawaban', 'konsekuensi']
    # Only take needed cols from df_lampiran, rename id to lampiran_id to avoid conflicts
    df_lampiran_slim = df_lampiran[['id'] + lampiran_cols].rename(columns={'id': 'lampiran_id'})
    df_skp_lampiran = df_skp[['nip', 'lampiran_id']].merge(df_lampiran_slim, on='lampiran_id', how='left')
    df_skp_lampiran['has_lampiran'] = (
        df_skp_lampiran[lampiran_cols].applymap(lambda x: isinstance(x, list) and len(x) > 0)
    ).all(axis=1)
    df_skp_lampiran_slim = df_skp_lampiran[['nip', 'has_lampiran']].drop_duplicates()

    # --- Ekspektasi Perilaku ---
    ekspektasi_cols = [
        'ekspektasi_berorientasi_pelayanan',
        'ekspektasi_akuntabel',
        'ekspektasi_kompeten',
        'ekspektasi_harmonis',
        'ekspektasi_loyal',
        'ekspektasi_adaptif',
        'ekspektasi_kolaboratif',
    ]
    umpan_balik_cols = [
        'umpan_balik_berorientasi_pelayanan',
        'umpan_balik_akuntabel',
        'umpan_balik_kompeten',
        'umpan_balik_harmonis',
        'umpan_balik_loyal',
        'umpan_balik_adaptif',
        'umpan_balik_kolaboratif',
    ]

    # Only take needed cols from df_ekspektasi, rename id to ekspektasi_id to avoid conflicts
    df_ekspektasi_slim = df_ekspektasi[['id'] + ekspektasi_cols + umpan_balik_cols].rename(columns={'id': 'ekspektasi_id'})
    df_skp_ekspektasi = df_skp[['nip', 'ekspektasi_id']].merge(df_ekspektasi_slim, on='ekspektasi_id', how='left')
    df_skp_ekspektasi['has_ekspektasi_perilaku'] = df_skp_ekspektasi[ekspektasi_cols].notna().any(axis=1)
    df_skp_ekspektasi['has_ub_perilaku'] = df_skp_ekspektasi[umpan_balik_cols].notna().any(axis=1)
    df_skp_ekspektasi_slim = df_skp_ekspektasi[['nip', 'has_ekspektasi_perilaku', 'has_ub_perilaku']].drop_duplicates()

    # --- Recap ---
    df_skp_recap = df_skp.merge(df_skp_count, on='nip', how='left')
    df_skp_recap = df_skp_recap.merge(df_skp_lampiran_slim, on='nip', how='left')
    df_skp_recap = df_skp_recap.merge(df_skp_ekspektasi_slim, on='nip', how='left')

    df_skp_recap['tanggal_export'] = pd.to_datetime(df_skp_recap['tanggal_export']).dt.date
    df_skp_recap['exported'] = df_skp_recap['generated_url'].notna()

    df_skp_recap_slim = df_skp_recap[['nip', 'tanggal_tutup', 'nip_atasan_skp', 'count_sasaran', 'tanggal_export', 'exported', 'all_realisasi_valid', 'has_ekspektasi_perilaku', 'has_ub_perilaku', 'has_ub_kinerja', 'has_lampiran']].drop_duplicates()

    # Final data combined with monitoring table
    df_skp_final = df_monitoring.merge(df_skp_recap_slim, left_on='peg_nip', right_on='nip', how='left')

    df_skp_final = df_skp_final.drop(columns=['nip'], errors='ignore')

    return df_skp_final


In [7]:
df_mon_at = pd.DataFrame()
df_mon_at = build_monitoring_table(df_vpd_akhir_tahun, engine_2025, 'tahunan')

df_mon_tw4 = pd.DataFrame()
df_mon_tw4 = build_monitoring_table(df_vpd_tw4, engine_2025, 'triwulan_4')


In [8]:
try:
    engine_2025.dispose()
except:
    pass

# Recreate the engine
engine_2025 = create_engine(connection_string, echo=False)

In [8]:
# Export to xlsx
import datetime
df_mon_at.to_excel(f'{datetime.datetime.now().strftime("%Y%m%d")} Monitoring SKP Akhir Tahun.xlsx', index=False)
df_mon_tw4.to_excel(f'{datetime.datetime.now().strftime("%Y%m%d")} Monitoring SKP Triwulan 4.xlsx', index=False)